# Additional End of week Exercise - week 2

Now use everything you've learned from Week 2 to build a full prototype for the technical question/answerer you built in Week 1 Exercise.

This should include a Gradio UI, streaming, use of the system prompt to add expertise, and the ability to switch between models. Bonus points if you can demonstrate use of a tool!

If you feel bold, see if you can add audio input so you can talk to it, and have it respond with audio. ChatGPT or Claude can help you, or email me if you have questions.

I will publish a full solution here soon - unless someone beats me to it...

There are so many commercial applications for this, from a language tutor, to a company onboarding solution, to a companion AI to a course (like this one!) I can't wait to see your results.

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
from tavily import TavilyClient
import json

In [ ]:
MODEL_GPT = "gpt-4.1-mini"
MODEL_GPT_KEY = "GPT"
MODEL_LLAMA = "llama3.2:1b"
MODEL_LLAMA_KEY = "LLAMA"
OLLAMA_BASE_URL = "http://localhost:11434/v1"

In [ ]:
load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
# Initailize providers
openai = OpenAI()
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")

In [ ]:
system_message ="You are a helpful assistant that takes a technical question and responds with an explanation."

In [ ]:
tavily_client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))

def search_with_tavily(query):
    print(f"search_with_tavily called with query: {query}")
    results = tavily_client.search(query=query, search_depth="advanced", limit=3)
    return "\n\n".join(
        f"**{r['title']}**\n{r['content']}\nSource: {r['url']}"
        for r in results.get("results", [])
    )

tavily_search = {
    "name": "tavily_search",
    "description": "Search the web for latest technical information, docs, release notes, or bug fixes.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {"type": "string", "description": "The search query"}
        },
        "required": ["query"]
    }
}

tools = [{"type": "function", "function": tavily_search}]

In [ ]:
def complete_with_gpt(messages, history, tools):
    while True:
        stream = openai.chat.completions.create(
            model=MODEL_GPT,
            messages=messages,
            tools=tools,
            stream=True
        )

        tool_calls_acc = {}
        result = ""
        finish_reason = None

        for chunk in stream:
            delta = chunk.choices[0].delta
            finish_reason = chunk.choices[0].finish_reason

            # Accumulate tool call chunks
            if delta.tool_calls:
                for tc in delta.tool_calls:
                    idx = tc.index
                    if idx not in tool_calls_acc:
                        tool_calls_acc[idx] = {"id": tc.id, "name": "", "arguments": ""}
                    if tc.function.name:
                        tool_calls_acc[idx]["name"] += tc.function.name
                    if tc.function.arguments:
                        tool_calls_acc[idx]["arguments"] += tc.function.arguments

            elif delta.content:
                result += delta.content
                yield history + [{"role": "assistant", "content": result}]

        # If tool call requested, execute and loop
        if finish_reason == "tool_calls" and tool_calls_acc:
            tool_call_list = []
            tool_results = []

            for tc in tool_calls_acc.values():
                args = json.loads(tc["arguments"])
                tool_call_list.append({
                    "id": tc["id"],
                    "type": "function",
                    "function": {"name": tc["name"], "arguments": tc["arguments"]}
                })
                tool_results.append({
                    "role": "tool",
                    "tool_call_id": tc["id"],
                    "content": search_with_tavily(args["query"])
                })

            messages.append({"role": "assistant", "content": None, "tool_calls": tool_call_list})
            messages.extend(tool_results)
            # Loop back to get final answer
        else:
            break

def complete_with_llama(messages, history):
    stream = ollama.chat.completions.create(
        model=MODEL_LLAMA,
        messages=messages,
        stream=True
    )

    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield history + [{"role": "assistant", "content": result}]

def chat(history, model):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history

    if model == MODEL_LLAMA_KEY:
        yield from complete_with_llama(messages, history)
    else:
        yield from complete_with_gpt(messages, history, tools)


In [ ]:
# This is a helper function to put the user message into the chatbot history before we call the chat function.
def put_message_in_chatbot(message, history):
        return "", history + [{"role":"user", "content":message}]

# UI definition

with gr.Blocks() as ui:
    with gr.Row():
        chatbot = gr.Chatbot(height=500, type="messages")
    with gr.Row():
         model_selector = gr.Dropdown([MODEL_GPT_KEY, MODEL_LLAMA_KEY], label="Select model", value=MODEL_GPT_KEY)
    # with gr.Row():
    #     audio_output = gr.Audio(autoplay=True)
    with gr.Row():
        message = gr.Textbox(label="Chat with our AI Assistant:")

# Hooking up events to callbacks

    message.submit(put_message_in_chatbot, inputs=[message, chatbot], outputs=[message, chatbot]).then(
        chat, inputs=[chatbot, model_selector], outputs=[chatbot]
    )

ui.launch(inbrowser=True)